# 1D Heat Equation: Preliminary example

In this notebook we illustrate the performance of the explicit finite differences method for solving the 1D heat equation
\begin{align*}
\frac{\partial u}{\partial t} &= \frac{\partial^2 u}{\partial x^2}\\
u(0,t)&=0\\
u(1,t)&=0\\
u(x,0)&=f(x)
\end{align*}
for $x\in(0,1)$ and $t\ge0$. We set the diffusion coefficient $D=1$ and the initial condition $f(x)=1-4(x-1/2)^2$, and compare our numerical approximations with the exact solution.

We also set the parameters for the numerical schemes as follows:

- Spatial step-size: $\Delta x = 1/10$.
- Temporal step-size: $\Delta t = 0.001$.
- Stability parameter: $s=\frac{D\Delta t}{(\Delta x)^2}=0.1$.

We will compare the approximate solution with the exact solution at $t=0.01$. We also show an animation of the change in temperature through the given time interval using the exact solution and the approximations.

## Libraries

In [1]:
import numpy as np
from numpy.linalg import solve, norm
from numpy import pi,exp,sin,cos
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.animation as animation
import pandas as pd

## $t\in [0, 0.01]$

### Exact solution

In [2]:
uexmp = np.loadtxt("uexmp.csv", delimiter=",")
uexmp = uexmp.T

FileNotFoundError: uexmp.csv not found.

### Discretized exact solution

In [ ]:
uexmp1 = np.loadtxt("uexmp1.csv", delimiter=",")
uexmp1 = uexmp1.T

### Parameter setup

In [ ]:
xmin, xmax = 0, 1
N = 10
dx = (xmax-xmin)/N
x = np.linspace(xmin,xmax,N+1)

tmin, tmax = 0, 0.01
dt = 0.001
M = int((tmax-tmin)/dt)
t = np.linspace(tmin,tmax,M+1)

D = 1
s = D*dt/(dx*dx)

X,T = np.meshgrid(x,t)

In [ ]:
(xmin,xmax),(tmin,tmax),N,M,D,dx,dt,s

### Initial condition

In [ ]:
def f(x):
    return 1-4*(x-1/2)**2

### Initial graphs

In [ ]:
xx = np.linspace(xmin,xmax,num=101)
tt = np.linspace(tmin,tmax,num=101)
XX,TT = np.meshgrid(xx,tt)

In [ ]:
fig,ax = plt.subplots(subplot_kw={"projection": "3d"},figsize=(10,10), layout='tight')
ax.plot_surface(XX, TT, uexmp.T,cmap=cm.viridis)
ax.set(title='Exact solution',xlabel='x',ylabel='t',zlabel='u')
ax.set_box_aspect(None, zoom=0.87)
plt.show()

In [ ]:
fig,ax = plt.subplots()
ct = ax.contourf(XX, TT, uexmp.T);
fig.colorbar(ct)
ax.set(title='Temperature contour map',xlabel='x',ylabel='t')
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(xx,f(xx));
ax.set(title="Initial condition",xlabel='x',ylabel='u(x,0)')
ax.grid()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,f(x));
ax.set(title="Discretized initial condition",xlabel='x',ylabel='u(x,0)')
ax.grid()
plt.show()

In [ ]:
np.shape(uexmp)

In [ ]:
fig,ax = plt.subplots()
for k in range(0,6):
    ax.plot(xx,uexmp[:,20*k],label=f't = {t[2*k]}')
ax.set(title='Temperature variation over time',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

### Animation of exact solution

In [ ]:
fig, ax = plt.subplots()
heat_exact, = ax.plot(xx,uexmp[:,0]);

xlim = (-0.1, 1.1)
ylim = (-0.1, 1.1)
ax.set(xlabel='x',ylabel='u(x,t)',xlim=xlim,ylim=ylim)
ax.grid()

def animate(k):
    heat_exact.set_ydata(uexmp[:,k])
    ax.set(title=f"Exact solution at t = {k*dt/10:.4f}")
    return heat_exact

ani = animation.FuncAnimation(fig, animate, frames=101)#, interval=500)

plt.show()

ani.save('heat_exmp_soln.mp4',fps=50)#, fps=30, extra_args=['-vcodec', 'libx264'])
ani.save('heat_exmp_soln.gif',fps=50)
print("Animation saved successfully as heat_exmp_soln (mp4 and gif)")

### Forward Euler

#### Computations

In [ ]:
u = np.zeros((N+1,M+1))
u[:,0] = f(x)

for k in range(0,M):
    u[1:-1,k+1] = u[1:-1,k]+s*(u[2:,k]-2*u[1:-1,k]+u[:-2,k])

In [ ]:
u_euler_fwd = u[:,:]

In [ ]:
error1 = u_euler_fwd[:,:]-uexmp1[:,:]

In [ ]:
print(f'Linf error = {norm(error1[:,M],ord=np.inf)}')

In [ ]:
df_euler_fwd = pd.DataFrame(u_euler_fwd.T, columns = x, index = t)

In [ ]:
df_euler_fwd

In [ ]:
df_euler_fwd_error = pd.DataFrame(error1.T, columns = x, index = t)

In [ ]:
df_euler_fwd_error

#### Graphs

In [ ]:
fig,ax = plt.subplots(subplot_kw={"projection": "3d"},figsize=(10,10), layout='tight')
ax.plot_surface(X, T, u_euler_fwd.T,cmap=cm.viridis)
ax.set(title='Forward Euler approximation',xlabel='x',ylabel='t',zlabel='u')
ax.set_box_aspect(None, zoom=0.87)
plt.show()

In [ ]:
fig,ax = plt.subplots()
ct = ax.contourf(X, T, u_euler_fwd.T);
fig.colorbar(ct)
ax.set(title='Temperature contour map',xlabel='x',ylabel='t')
plt.show()

In [ ]:
fig,ax = plt.subplots()
for k in range(0,M+1,M//5):
    ax.plot(x,u_euler_fwd[:,k],label=f't = {t[k]}')
ax.set(title='Temperature variation over time',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,u_euler_fwd[:,M],label=f'Approximation at t = {t[M]}')
ax.plot(x,uexmp1[:,M],label=f'Exact solution at t = {tmax}')
ax.set(title='Comparison between exact and approximate solution',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

In [ ]:
error1 = u_euler_fwd[:,:]-uexmp1[:,:]

In [ ]:
print(f'Linf error = {norm(error1[:,M],ord=np.inf)}')

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,abs(error1[:,M]))
ax.set(title='Absolute error',xlabel='x',ylabel='error')
ax.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots()

heat_approx, = ax.plot(x,u_euler_fwd[:,0],label='Forward Euler approximation');
heat_exact, = ax.plot(x,uexmp1[:,0],linestyle='-.',label='Exact solution');

ax.legend()
ax.set(xlabel='x',ylabel='u(x,t)',xlim=xlim,ylim=ylim)
ax.grid()

def animate(k):
    heat_approx.set_ydata(u_euler_fwd[:,k])
    heat_exact.set_ydata(uexmp1[:,k])
    ax.set(title=f"Exact solution vs Forward Euler approximation at t = {k*dt:.3f}")
    return heat_approx,heat_exact

ani = animation.FuncAnimation(fig, animate, frames=M+1)#, interval=500)

plt.show()

ani.save('heat_exmp1_fwd.mp4')#, fps=30, extra_args=['-vcodec', 'libx264'])
ani.save('heat_exmp1_fwd.gif')
print("Animation saved successfully as heat_exmp1_fwd (mp4 and gif)")

In [ ]:
fig, ax = plt.subplots()

error, = ax.plot(x,abs(error1[:,0]));

#ax.legend()
ax.set(xlabel='x',ylabel='error',xlim=xlim,ylim=[0,6.5e-4])
ax.grid()

def animate(k):
    error.set_ydata(abs(error1[:,k]))
    ax.set(title=f"Error propagation. t = {k*dt:.3f}")
    return error

ani = animation.FuncAnimation(fig, animate, frames=M+1)#, interval=500)

plt.show()

ani.save('heat_exmp1_fwd_error.mp4')#, fps=30, extra_args=['-vcodec', 'libx264'])
ani.save('heat_exmp1_fwd_error.gif')
print("Animation saved successfully as heat_exmp1_fwd_error (mp4 and gif)")

## $t\in [0, 0.1]$

### Exact solution

In [ ]:
uexmpb = np.loadtxt("uexmpb.csv", delimiter=",")
uexmpb = uexmpb.T

### Discretized exact solution

In [ ]:
uexmp2 = np.loadtxt("uexmp2.csv", delimiter=",")
uexmp2 = uexmp2.T

### Parameter setup

In [ ]:
xmin, xmax = 0, 1
N = 10
dx = (xmax-xmin)/N
x = np.linspace(xmin,xmax,N+1)

tmin, tmax = 0, 0.1
dt = 0.001
M = int((tmax-tmin)/dt)
t = np.linspace(tmin,tmax,M+1)

D = 1
s = D*dt/(dx*dx)

X,T = np.meshgrid(x,t)

In [ ]:
(xmin,xmax),(tmin,tmax),N,M,D,dx,dt,s

### Initial condition

In [ ]:
def f(x):
    return 1-4*(x-1/2)**2

### Initial graphs

In [ ]:
xx = np.linspace(xmin,xmax,num=101)
tt = np.linspace(tmin,tmax,num=1001)
XX,TT = np.meshgrid(xx,tt)

In [ ]:
fig,ax = plt.subplots(subplot_kw={"projection": "3d"},figsize=(10,10), layout='tight')
ax.plot_surface(XX, TT, uexmpb.T,cmap=cm.viridis)
ax.set(title='Exact solution',xlabel='x',ylabel='t',zlabel='u')
ax.set_box_aspect(None, zoom=0.87)
plt.show()

In [ ]:
fig,ax = plt.subplots()
ct = ax.contourf(XX, TT, uexmpb.T);
fig.colorbar(ct)
ax.set(title='Temperature contour map',xlabel='x',ylabel='t')
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(xx,f(xx));
ax.set(title="Initial condition",xlabel='x',ylabel='u(x,0)')
ax.grid()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,f(x));
ax.set(title="Discretized initial condition",xlabel='x',ylabel='u(x,0)')
ax.grid()
plt.show()

In [ ]:
np.shape(uexmpb)

In [ ]:
fig,ax = plt.subplots()
for k in range(0,6):
    ax.plot(xx,uexmpb[:,200*k],label=f't = {t[20*k]}')
ax.set(title='Temperature variation over time',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

### Animation of exact solution

In [ ]:
fig, ax = plt.subplots()
heat_exact, = ax.plot(xx,uexmpb[:,0]);

xlim = (-0.1, 1.1)
ylim = (-0.1, 1.1)
ax.set(xlabel='x',ylabel='u(x,t)',xlim=xlim,ylim=ylim)
ax.grid()

def animate(k):
    heat_exact.set_ydata(uexmpb[:,10*k])
    ax.set(title=f"Exact solution at t = {k*dt/10:.4f}")
    return heat_exact

ani = animation.FuncAnimation(fig, animate, frames=101)#, interval=500)

plt.show()

ani.save('heat_exmpb_soln.mp4',fps=50)#, fps=30, extra_args=['-vcodec', 'libx264'])
ani.save('heat_exmpb_soln.gif',fps=50)
print("Animation saved successfully as heat_exmpb_soln (mp4 and gif)")

### Forward Euler

#### Computations

In [ ]:
u = np.zeros((N+1,M+1))
u[:,0] = f(x)

for k in range(0,M):
    u[1:-1,k+1] = u[1:-1,k]+s*(u[2:,k]-2*u[1:-1,k]+u[:-2,k])

In [ ]:
u_euler_fwd = u[:,:]

In [ ]:
error2 = u_euler_fwd[:,:]-uexmp2[:,:]

In [ ]:
print(f'Linf error = {norm(error2[:,M],ord=np.inf)}')

In [ ]:
df_euler_fwd = pd.DataFrame(u_euler_fwd.T, columns = x, index = t)

In [ ]:
df_euler_fwd

In [ ]:
df_euler_fwd_error = pd.DataFrame(error2.T, columns = x, index = t)

In [ ]:
df_euler_fwd_error

#### Graphs

In [ ]:
fig,ax = plt.subplots(subplot_kw={"projection": "3d"},figsize=(10,10), layout='tight')
ax.plot_surface(X, T, u_euler_fwd.T,cmap=cm.viridis)
ax.set(title='Forward Euler approximation',xlabel='x',ylabel='t',zlabel='u')
ax.set_box_aspect(None, zoom=0.87)
plt.show()

In [ ]:
fig,ax = plt.subplots()
ct = ax.contourf(X, T, u_euler_fwd.T);
fig.colorbar(ct)
ax.set(title='Temperature contour map',xlabel='x',ylabel='t')
plt.show()

In [ ]:
fig,ax = plt.subplots()
for k in range(0,M+1,M//5):
    ax.plot(x,u_euler_fwd[:,k],label=f't = {t[k]}')
ax.set(title='Temperature variation over time',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,u_euler_fwd[:,M],label=f'Approximation at t = {t[M]}')
ax.plot(x,uexmp2[:,M],label=f'Exact solution at t = {tmax}')
ax.set(title='Comparison between exact and approximate solution',xlabel='x',ylabel='u')
ax.grid()
ax.legend()
plt.show()

In [ ]:
error2 = u_euler_fwd[:,:]-uexmp2[:,:]

In [ ]:
print(f'Linf error = {norm(error2[:,M],ord=np.inf)}')

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,abs(error2[:,M]))
ax.set(title='Absolute error',xlabel='x',ylabel='error')
ax.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots()

heat_approx, = ax.plot(x,u_euler_fwd[:,0],label='Forward Euler approximation');
heat_exact, = ax.plot(x,uexmp2[:,0],linestyle='-.',label='Exact solution');

ax.legend()
ax.set(xlabel='x',ylabel='u(x,t)',xlim=xlim,ylim=ylim)
ax.grid()

def animate(k):
    heat_approx.set_ydata(u_euler_fwd[:,k])
    heat_exact.set_ydata(uexmp2[:,k])
    ax.set(title=f"Exact solution vs Forward Euler approximation at t = {k*dt:.3f}")
    return heat_approx,heat_exact

ani = animation.FuncAnimation(fig, animate, frames=M+1)#, interval=500)

plt.show()

ani.save('heat_exmp2_fwd.mp4', fps=50)#, fps=30, extra_args=['-vcodec', 'libx264'])
ani.save('heat_exmp2_fwd.gif', fps=50)
print("Animation saved successfully as heat_exmp2_fwd (mp4 and gif)")

In [ ]:
fig, ax = plt.subplots()

error, = ax.plot(x,abs(error2[:,0]));

#ax.legend()
ax.set(xlabel='x',ylabel='error',xlim=xlim,ylim=[0,1.5e-3])
ax.grid()

def animate(k):
    error.set_ydata(abs(error2[:,k]))
    ax.set(title=f"Error propagation. t = {k*dt:.3f}")
    return error

ani = animation.FuncAnimation(fig, animate, frames=M+1)#, interval=500)

plt.show()

ani.save('heat_exmp2_fwd_error.mp4', fps=50)#, fps=30, extra_args=['-vcodec', 'libx264'])
ani.save('heat_exmp2_fwd_error.gif', fps=50)
print("Animation saved successfully as heat_exmp2_fwd_error (mp4 and gif)")